In [1]:
#export
import json,re
from pathlib import Path
import glob

In [2]:
#export
def getSortedFiles(path):
    print(path+'/**/*.ipynb')
    ret =  glob.glob(path+'/**/*.ipynb', recursive=True) 
    print(ret)
    if 0==len(ret): 
        print('WARNING: No files found')
        return ret
    return sorted(ret)

In [3]:
#export
def is_excludeFile(cell):
    if cell['cell_type'] != 'code': return False
    src = cell['source']
    if len(src) == 0 or len(src[0]) < 7: return False
    #import pdb; pdb.set_trace()
    return re.match(r'^\s*#\s*excludefile\s*$', src[0], re.IGNORECASE) is not None

def is_export(cell):
    if cell['cell_type'] != 'code': return False
    src = cell['source']
    if len(src) == 0 or len(src[0]) < 7: return False
    #import pdb; pdb.set_trace()
    return re.match(r'^\s*#\s*export\s*$', src[0], re.IGNORECASE) is not None

def notebook2script(fname,outpath='src'):
    "Finds cells starting with `#export` and puts them into a new module"
    fname = Path(fname)
    fname_out = f'{fname.stem.split(".")[0]}.py'
    print('Single...',fname, ' en ', fname_out)
    main_dic = json.load(open(fname,'r',encoding="utf-8"))
    excludeFile=(main_dic['cells'][0]['source'][0]).lower()
    if (excludeFile=='#excludefile'):
        return
    code_cells = [c for c in main_dic['cells'] if is_export(c)]
    module = f'''
#################################################
### THIS FILE WAS AUTOGENERATED! DO NOT EDIT! ###
#################################################
# file to edit: {fname}
'''
    for cell in code_cells: module += ''.join(cell['source'][1:]) + '\n\n'
    # remove trailing spaces
    module = re.sub(r' +$', '', module, flags=re.MULTILINE)
    #output_path = fname.parent
    output_path = outpath #+str(output_path)[3:]
    Path(output_path).mkdir(parents=True, exist_ok=True)
    output_path=Path(output_path)/fname_out
    open(output_path,'w', encoding="utf-8").write(module[:-2])
    #print(f"notebook2script {fname} to {output_path}")

    
def notebooks2script(path='nbs',outpath='src'):
    print('notebooks2script...')
    [notebook2script(f,outpath) for f in getSortedFiles(path)]
    print('...End')    

In [4]:
#export
def clean_cell(cell):
    if 'execution_count' in cell: cell['execution_count'] = None
    if 'outputs' in cell: cell['outputs'] = []
    if cell['source'] == ['']: cell['source'] = []
    cell['metadata'] = {} 

def notebookClean(fname):    
    print(f"notebookClean {fname}")
    nb = json.loads(open(fname, 'r', encoding='utf-8').read())
    for c in nb['cells']: clean_cell(c)
    x = json.dumps(nb, sort_keys=True, indent=1, ensure_ascii=False)
    with open(fname, 'w', encoding='utf-8') as f:
        f.write(x)
        f.write("\n")

def notebooksClean(path='nbs'):
    print('notebooksClean ...')
    [notebookClean(f) for f in getSortedFiles(path)]
    print('...End')        

In [5]:
notebooks2script('transcribe','transcribe')



notebooks2script...
transcribe/**/*.ipynb
['transcribe/transcribe.ipynb', 'transcribe/incidencias_manager.ipynb', 'transcribe/deepresearch_youtube.ipynb', 'transcribe/cendilu.ipynb', 'transcribe/logs.ipynb', 'transcribe/ollama_manager.ipynb', 'transcribe/incidencias_excel.ipynb', 'transcribe/mail_manager.ipynb', 'transcribe/extrae_pdf.ipynb', 'transcribe/incidenciasAPP.ipynb', 'transcribe/file_manager.ipynb', 'transcribe/agente.ipynb', 'transcribe/requeriments.ipynb', 'transcribe/youtube_manager.ipynb', 'transcribe/parametros/prompts.ipynb']
Single... transcribe/agente.ipynb  en  agente.py
Single... transcribe/cendilu.ipynb  en  cendilu.py
Single... transcribe/deepresearch_youtube.ipynb  en  deepresearch_youtube.py
Single... transcribe/extrae_pdf.ipynb  en  extrae_pdf.py
Single... transcribe/file_manager.ipynb  en  file_manager.py
Single... transcribe/incidenciasAPP.ipynb  en  incidenciasAPP.py
Single... transcribe/incidencias_excel.ipynb  en  incidencias_excel.py
Single... transcribe/